# Bank Complaints Analysis: 01 Explore & Clean

**Goal:** turn the raw CFPB complaint file into a clean table of credit card and checking/savings complaints that have a narrative.

- **Source:** CFPB Consumer Complaint Database, narratives archive (FOIA reading room)
- **Input:** `data/raw/CCDB_Export_14_November_2025_through_December_2025.csv` (1,009,132 complaints)
- **Output:** `data/processed/bank_clean.csv` (13,665 rows x 22 columns)

## 1. Check the newest file first (August 2026)

**Why:** before building anything, confirm the column I need (the narrative) actually has data.

**Result:** only **1 of 667,140** complaints has a narrative. The CFPB stopped publishing narratives on Aug 14, 2026, so this file can't support the project. I switched to an older archive file.

In [1]:
import pandas as pd

path = r"..\data\raw\CCDB_Export_21_August_2026.csv"
df = pd.read_csv(path, usecols=["Date received", "Product", "Consumer complaint narrative"], low_memory=False)

df.shape

(667140, 3)

In [2]:
df["Product"].value_counts()

Product
Credit reporting or other personal consumer reports        619520
Debt collection                                             20398
Credit card                                                  7996
Checking or savings account                                  7552
Money transfer, virtual currency, or money service           3093
Mortgage                                                     2714
Vehicle loan or lease                                        2001
Payday loan, title loan, personal loan, or advance loan      1654
Student loan                                                 1407
Prepaid card                                                  491
Debt or credit management                                     314
Name: count, dtype: int64

In [3]:
df["Consumer complaint narrative"].notna().sum()

np.int64(1)

In [4]:
df["Consumer complaint narrative"].isna().sum()

np.int64(667139)

## 2. Check the Nov–Dec 2025 archive file

**Result:** **134,641** narratives out of 1,009,132 complaints. This file works.

In [5]:
path2 = r"..\data\raw\CCDB_Export_14_November_2025_through_December_2025.csv"
df2 = pd.read_csv(path2, usecols=["Date received","Product","Consumer complaint narrative"], low_memory=False)

In [6]:
df2.shape

(1009132, 3)

In [7]:
df2["Consumer complaint narrative"].notna().sum()

np.int64(134641)

## 3. Filter to bank complaints with a narrative

**Why:** 90% of the file is credit-reporting complaints. My questions are about banks, so I keep only **Credit card** and **Checking or savings account** complaints that include a narrative.

**Result:** 1,009,132 → **13,665** rows. Saved as `bank_nov_dec_2025.csv` (the filtered slice, before cleaning).

In [8]:
df_full = pd.read_csv(path2, low_memory=False)
df_full.shape

(1009132, 16)

In [9]:
products = ["Credit card", "Checking or savings account"]
mask = df_full["Product"].isin(products) & df_full["Consumer complaint narrative"].notna()
bank = df_full[mask].copy()
bank.shape

(13665, 16)

In [10]:
del df_full

In [11]:
bank.to_csv(r"..\data\processed\bank_nov_dec_2025.csv",index=False)

## 4. Quality checks

**Why:** check missing values, duplicates and key values before trusting any number.

**Findings:**
- `Tags` blank in 11,263 rows: blank means *not tagged* (not a Servicemember / Older American complaint), not missing data.
- `Company public response` blank in 7,663 rows: not used in my questions, left as is.
- `State` blank in 120 rows, `Sub-issue` in 18: small, left as is.
- **0 duplicate** Complaint IDs.
- 13,542 timely / 123 not timely.

In [12]:
bank.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13665 entries, 27 to 1009101
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Date received                 13665 non-null  object
 1   Product                       13665 non-null  object
 2   Sub-product                   13665 non-null  object
 3   Issue                         13665 non-null  object
 4   Sub-issue                     13647 non-null  object
 5   Consumer complaint narrative  13665 non-null  object
 6   Company public response       6002 non-null   object
 7   Company                       13665 non-null  object
 8   State                         13545 non-null  object
 9   ZIP code                      13665 non-null  object
 10  Tags                          2402 non-null   object
 11  Submitted via                 13665 non-null  object
 12  Date sent to company          13665 non-null  object
 13  Company response t

In [13]:
bank.isna().sum()

Date received                       0
Product                             0
Sub-product                         0
Issue                               0
Sub-issue                          18
Consumer complaint narrative        0
Company public response          7663
Company                             0
State                             120
ZIP code                            0
Tags                            11263
Submitted via                       0
Date sent to company                0
Company response to consumer        0
Timely response?                    0
Complaint ID                        0
dtype: int64

In [14]:
bank["Complaint ID"].duplicated().sum()

np.int64(0)

In [15]:
bank["Timely response?"].value_counts()

Timely response?
Yes    13542
No       123
Name: count, dtype: int64

In [16]:
bank["Company response to consumer"].value_counts()

Company response to consumer
Closed with explanation            10564
Closed with monetary relief         2011
Closed with non-monetary relief     1068
Untimely response                     22
Name: count, dtype: int64

## 5. Clean and add columns

- Column names → lowercase with underscores (easier to type, SQL-friendly).
- Date columns → real dates.
- New columns: `month`, `days_to_company`, `narrative_length`, and 1/0 flags `is_timely`, `got_relief`, `got_money` (the mean of a 1/0 flag = a percentage).

**Result:** 13,665 rows x 22 columns, saved as `bank_clean.csv`.

*Note: CSV doesn't keep data types. When reloading `bank_clean.csv`, use `parse_dates=["date_received", "date_sent_to_company"]`.*

In [17]:
bank.columns = (bank.columns
               .str.lower().str.replace(" ","_").str.replace("-","_").str.replace("?","",regex=False))
bank.columns

Index(['date_received', 'product', 'sub_product', 'issue', 'sub_issue',
       'consumer_complaint_narrative', 'company_public_response', 'company',
       'state', 'zip_code', 'tags', 'submitted_via', 'date_sent_to_company',
       'company_response_to_consumer', 'timely_response', 'complaint_id'],
      dtype='object')

In [18]:
bank = bank.rename(columns={"consumer_complaint_narrative": "narrative"})

In [19]:
bank["date_received"] = pd.to_datetime(bank["date_received"])
bank["date_sent_to_company"]= pd.to_datetime(bank["date_sent_to_company"])

In [20]:
bank["month"]= bank["date_received"].dt.to_period("M")
bank["days_to_company"]= (bank["date_sent_to_company"] - bank["date_received"]).dt.days
bank["narrative_length"]= bank["narrative"].str.len()
bank["is_timely"]= (bank["timely_response"]=="Yes").astype(int)
bank["got_relief"]= bank["company_response_to_consumer"].isin(["Closed with monetary relief", "Closed with non-monetary relief"]).astype(int)
bank["got_money"] = (bank["company_response_to_consumer"]== "Closed with monetary relief").astype(int)

In [21]:
bank[["month", "days_to_company","narrative_length","is_timely","got_relief","got_money"]].describe()

,days_to_company,narrative_length,is_timely,got_relief,got_money
count,13665.000000,13665.000000,13665.000000,13665.000000,13665.000000
mean,2.456787,1343.725576,0.990999,0.225320,0.147164
std,13.663489,1271.069933,0.094450,0.417808,0.354283
min,0.000000,49.000000,0.000000,0.000000,0.000000
25%,0.000000,554.000000,1.000000,0.000000,0.000000
50%,0.000000,1045.000000,1.000000,0.000000,0.000000
75%,0.000000,1723.000000,1.000000,0.000000,0.000000
max,181.000000,23901.000000,1.000000,1.000000,1.000000


In [22]:
bank.to_csv(r"..\data\processed\bank_clean.csv",index=False)

## 6. First look

**Findings:**
- Most complaints: JPMorgan Chase (1,274), Capital One (1,227), Citibank (1,140).
- Top issue: **Managing an account** (3,836).
- Relief rate varies a lot by bank: Citibank 44.5%, Chase 16.1%, Block (Cash App) 0%.
- Narrative length is about the same with or without relief (median 1,038 vs 1,058 characters).

In [23]:
bank["company"].value_counts().head(10)

company
JPMORGAN CHASE & CO.                     1274
CAPITAL ONE FINANCIAL CORPORATION        1227
CITIBANK, N.A.                           1140
BANK OF AMERICA, NATIONAL ASSOCIATION     989
Chime Financial Inc                       961
WELLS FARGO & COMPANY                     901
SYNCHRONY FINANCIAL                       776
Block, Inc.                               419
AMERICAN EXPRESS COMPANY                  416
U.S. BANCORP                              347
Name: count, dtype: int64

In [24]:
bank["issue"].value_counts().head(8)

issue
Managing an account                                             3836
Problem with a purchase shown on your statement                 2060
Problem with a lender or other company charging your account    1095
Closing an account                                               920
Fees or interest                                                 837
Other features, terms, or problems                               814
Getting a credit card                                            741
Closing your account                                             570
Name: count, dtype: int64

In [25]:
top = bank["company"].value_counts().head(10).index
(bank[bank["company"].isin(top)]
 .groupby("company")[["got_relief", "is_timely"]]
 .mean()
 .sort_values("got_relief", ascending=False))

,got_relief,is_timely
company,,
"CITIBANK, N.A.",0.444737,1.000000
"BANK OF AMERICA, NATIONAL ASSOCIATION",0.386249,0.997978
AMERICAN EXPRESS COMPANY,0.276442,0.947115
WELLS FARGO & COMPANY,0.233074,1.000000
SYNCHRONY FINANCIAL,0.224227,0.990979
U.S. BANCORP,0.193084,1.000000
CAPITAL ONE FINANCIAL CORPORATION,0.169519,1.000000
JPMORGAN CHASE & CO.,0.160911,1.000000
Chime Financial Inc,0.125911,1.000000


In [26]:
bank.groupby("got_relief")["narrative_length"].median()

got_relief
0    1038.0
1    1058.0
Name: narrative_length, dtype: float64

In [27]:
print(bank["narrative"].iloc[0])

I have contacted XXXX multiple times regarding receiving money from my account. My account was suspended and subsequently terminated for failure to provide identification. I provided identification on multiple occasions. I requested a new account specialist to review the account. I was told XXXX could legally hold my money for up to XXXX days after closing/suspending the account. The User Agreement does confirm this ( see attached ) if any of my usage violates the " Acceptable Use Policy '' ( also attached ). XXXX has not provided any reason written kr verbal of how my User has violated the " Acceptable Use Policy ''. I just want my money. I am hungry.


## 8. Sample for AI labelling (Q5)

In [28]:
mac = bank[bank["issue"] == "Managing an account"]
sample = mac.sample(200, random_state=42)[["complaint_id","narrative"]].copy()
sample["my_label"]= ""
sample.to_csv(r"..\data\processed\label_sample_200.csv", index=False)
sample.shape

(200, 3)

In [29]:
sample.to_excel(r"..\data\processed\label_sample_200.xlsx", index=False)
